# Programación de Sistemas de Base I - Inspección de la Estructura de un Compilador y Procesamiento de Lenguajes
**Asignatura:** Programación de Sistemas de Base I (8.º Semestre)  
**Unidad I:** Introducción a la Compilación  
**Tiempo estimado:** 3 horas (Asíncrono / Guiado)  
**Repositorio Base / Entrega:** Integración con Moodle / GitHub

---


## 1. Objetivos de Aprendizaje
1. Identificar de forma tangible la diferencia entre ejecución compilada e interpretada utilizando las herramientas nativas del entorno (`gcc`, `python3`, `dis`, `ast`).
2. Explorar y visualizar la fase de Análisis Léxico y Sintáctico mediante la inspección del Árbol de Sintaxis Abstracta (AST) y la generación de Bytecode.
3. Comprender el rol de la Tabla de Símbolos en el seguimiento de identificadores durante el proceso de traducción.

---


## 2. Conexión Teórica (NotebookLM)
**Antes de continuar:** Accede al espacio de **NotebookLM del Curso** y realiza las siguientes preguntas de verificación:

- *¿Cuáles son las fases de la etapa de Análisis (Front-End) de un compilador?*  
**RESPUESTA:**

1. Análisis Lexicográfico (Léxico o Scanner): Lee el programa fuente carácter por carácter de izquierda a derecha y agrupa esos caracteres en unidades lógicas con significado llamadas tokens (como identificadores de usuario, palabras reservadas, constantes y operadores).
2. Análisis Sintáctico (Sintaxis o Parser): Recibe la secuencia de tokens generada por la fase léxica y comprueba que su orden y estructura cumplan estrictamente con las reglas gramaticales del lenguaje de programación.
Resultado: Construye una estructura jerárquica que representa la gramática del programa, que suele ser un árbol sintáctico o árbol de análisis sintáctico (AST).
3. Análisis Semántico (Semántica): Revisa el árbol sintáctico junto con la tabla de símbolos para asegurar que las instrucciones tengan un significado lógico y coherente respecto a las reglas del lenguaje.
Funciones clave: Su componente más importante es la comprobación (verificación) de tipos (por ejemplo, asegurar que no intentes sumar un texto con un número flotante si el lenguaje no lo permite) y la validación de que las variables utilizadas hayan sido declaradas previamente.

- *¿Qué diferencia a un Lexema de un Token?*  
**RESPUESTA:**

La diferencia fundamental radica en que un lexema es la cadena de caracteres concreta (el texto tal cual aparece en tu código), mientras que un token es la categoría abstracta en la que el compilador clasifica a esa cadena.

- *¿Qué función cumple la Tabla de Símbolos durante la compilación?*  
**RESPUESTA:**

La Tabla de Símbolos (también conocida como tabla de nombres o diccionario de datos) es una estructura de datos auxiliar y central que acompaña a prácticamente todas las fases de la compilación. Su propósito principal es actuar como un repositorio centralizado que asocia información detallada a los nombres (identificadores) que el programador define en el código fuente (como variables, constantes, tipos, funciones y procedimientos).


---

## 3. Sección Práctica 1: Pipeline de Compilación vs. Interpretación (CLI / Bash)
En esta sección analizaremos cómo el sistema operativo y el entorno procesan un lenguaje compilado (C) frente a uno interpretado/híbrido (Python).

### [CÓDIGO 1.1] Inspección de Herramientas del Sistema
```bash
!echo "=== COMPILADOR C DE LINUX (GCC) ==="
!gcc --version | head -n 1
!echo ""
!echo "=== INTÉRPRETE DE PYTHON ==="
!python3 --version

In [ ]:
!echo "=== COMPILADOR C DE LINUX (GCC) ==="
!gcc --version | head -n 1
!echo ""
!echo "=== INTÉRPRETE DE PYTHON ==="
!python3 --version

=== COMPILADOR C DE LINUX (GCC) ===
gcc (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0

=== INTÉRPRETE DE PYTHON ===
Python 3.13.15


### [CÓDIGO 1.2] Creación, Compilación y Análisis de Binario en C
```c
%%writefile hola_compilador.c
#include <stdio.h>
#define inputMsj "Escriba un numero entero: "
#define msj "La suma de %d + %d es: %d\n"

int sumar(int x, int y){
  return x + y;
}

int main() {
    int a = 5;
    int b;
    
    printf(inputMsj);
    scanf("%d",&b);
    int suma = sumar(a, b);
    printf(msj, a, b, suma);
    return 0;
}

In [ ]:
%%writefile hola_compilador.c
# include <stdio.h>
# define inputMsj "Escriba un numero entero: "
# define msj "La suma de %d + %d es: %d\n"

int sumar(int x, int y){
  return x + y;
}

int main() {
    int a = 5;
    int b;

    printf(inputMsj);
    scanf("%d",&b);
    int suma = sumar(a, b);
    printf(msj, a, b, suma);
    return 0;
}

Writing hola_compilador.c


## El Preprocesador (Antes del ensamblador)
Esta etapa expande las macros (como `#define`) e incluye los archivos de cabecera (como `#include <stdio.h>`).
```bash
!gcc -E hola_compilador.c -o hola_compilador.i
!echo === CÓDIGO PREPROCESADO ===
!cat hola_compilador.i | tail -n 20


In [ ]:
!gcc -E hola_compilador.c -o hola_compilador.i
!echo === CÓDIGO PREPROCESADO ===
!cat hola_compilador.i | tail -n 20

=== CÓDIGO PREPROCESADO ===
# 2 "hola_compilador.c" 2




# 5 "hola_compilador.c"
int sumar(int x, int y){
  return x + y;
}

int main() {
    int a = 5;
    int b;

    printf("Escriba un numero entero: ");
    scanf("%d",&b);
    int suma = sumar(a, b);
    printf("La suma de %d + %d es: %d\n", a, b, suma);
    return 0;
}


## Representación Intermedia (IR) y Optimizaciones
GCC utiliza una representación intermedia llamada GIMPLE antes de generar el ensamblador. Puedes ver cómo cambia el código antes y después de que el optimizador trabaje.

```bash
!gcc -O2 -fdump-tree-gimple hola_compilador.c -c
!echo === REPRESENTACIÓN INTERMEDIA (GIMPLE) ===
!cat hola_compilador.c.*gimple

In [ ]:
!gcc -O2 -fdump-tree-gimple hola_compilador.c -c
!echo === REPRESENTACIÓN INTERMEDIA (GIMPLE) ===
!cat hola_compilador.c.*gimple

hola_compilador.c: In function ‘main’:
hola_compilador.c:14:5: warning: ignoring return value of ‘scanf’ declared with attribute ‘warn_unused_result’ []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wunused-result-Wunused-result]8;;]
   14 |     scanf("%d",&b);
      |     ^~~~~~~~~~~~~~
/bin/bash: -c: line 1: syntax error near unexpected token `('
/bin/bash: -c: line 1: `echo === REPRESENTACIÓN INTERMEDIA (GIMPLE) ==='
int sumar (int x, int y)
{
  int D.2555;

  D.2555 = x + y;
  return D.2555;
}


int main ()
{
  int D.2557;

  {
    int a;
    int b;
    int suma;

    try
      {
        a = 5;
        printf ("Escriba un numero entero: ");
        scanf ("%d", &b);
        b.0_1 = b;
        suma = sumar (a, b.0_1);
        b.1_2 = b;
        printf ("La suma de %d + %d es: %d\n", a, b.1_2, suma);
        D.2557 = 0;
        return D.2557;
      }
    finally
      {
        b = {CLOBBER};
      }
  }
  D.2557 = 0;
  return D.2557;
}


__attribute__((artifici

## Compilación generando código ensamblador intermediate (.s)
 El compilador traduce el código **sin optimizaciones** línea por línea directamente a la memoria RAM (la pila o stack)

```bash
!gcc -S hola_compilador.c -o hola_compilador.s
!echo "=== CÓDIGO ENSAMBLADOR GENERADO (SÍNTESIS) ==="
!cat hola_compilador.s | head -n 25

In [ ]:
!gcc -S hola_compilador.c -o hola_compilador.s
!echo "=== CÓDIGO ENSAMBLADOR GENERADO (SÍNTESIS) ==="
!cat hola_compilador.s | head -n 25

=== CÓDIGO ENSAMBLADOR GENERADO (SÍNTESIS) ===
	.file	"hola_compilador.c"
	.text
	.globl	sumar
	.type	sumar, @function
sumar:
.LFB0:
	.cfi_startproc
	endbr64
	pushq	%rbp
	.cfi_def_cfa_offset 16
	.cfi_offset 6, -16
	movq	%rsp, %rbp
	.cfi_def_cfa_register 6
	movl	%edi, -4(%rbp)
	movl	%esi, -8(%rbp)
	movl	-4(%rbp), %edx
	movl	-8(%rbp), %eax
	addl	%edx, %eax
	popq	%rbp
	.cfi_def_cfa 7, 8
	ret
	.cfi_endproc
.LFE0:
	.size	sumar, .-sumar
	.section	.rodata


Para producir una traducción del código **optimizada** usa

```bash
!gcc -O3 -S hola_compilador.c -o hola_conmpilador_opt.s
!echo "=== CÓDIGO ENSAMBLADOR OPTIMIZADO GENERADO (SÍNTESIS) ==="
!cat hola_compilador_opt.s | head -n 25

In [ ]:
!gcc -O3 -S hola_compilador.c -o hola_compilador_opt.s
!echo "=== CÓDIGO ENSAMBLADOR OPTIMIZADO GENERADO (SÍNTESIS) ==="
!cat hola_compilador_opt.s | head -n 25

hola_compilador.c: In function ‘main’:
hola_compilador.c:14:5: warning: ignoring return value of ‘scanf’ declared with attribute ‘warn_unused_result’ []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wunused-result-Wunused-result]8;;]
   14 |     scanf("%d",&b);
      |     ^~~~~~~~~~~~~~
=== CÓDIGO ENSAMBLADOR OPTIMIZADO GENERADO (SÍNTESIS) ===
	.file	"hola_compilador.c"
	.text
	.p2align 4
	.globl	sumar
	.type	sumar, @function
sumar:
.LFB23:
	.cfi_startproc
	endbr64
	leal	(%rdi,%rsi), %eax
	ret
	.cfi_endproc
.LFE23:
	.size	sumar, .-sumar
	.section	.rodata.str1.1,"aMS",@progbits,1
.LC0:
	.string	"Escriba un numero entero: "
.LC1:
	.string	"%d"
.LC2:
	.string	"La suma de %d + %d es: %d\n"
	.section	.text.startup,"ax",@progbits
	.p2align 4
	.globl	main
	.type	main, @function


## Superando limitaciones de la arquitectura fija (X86_64)
La máquina virtual de Colab corre sobre procesadores Intel o AMD de 64 bits.  
**Impacto:** El código ensamblador generado (.s) siempre estará en sintaxis AT&T (por defecto en Linux) y para arquitectura x86_64. Para ver código ensamblador con sintaxis de Intel (más parecida a la de Windows), agregar el parámetro ***-masm=intel*** a la orden de GCC:

```bash
!gcc -S -masm=intel hola_compilador.c -o hola_compilador_8664.s
!echo "=== CÓDIGO ENSAMBLADOR x86_64 GENERADO (SÍNTESIS) ==="
!cat hola_compilador_8664.s | head -n 25


In [ ]:
!gcc -S -masm=intel hola_compilador.c -o hola_compilador_8664.s
!echo "=== CÓDIGO ENSAMBLADOR x86_64 GENERADO (SÍNTESIS) ==="
!cat hola_compilador_8664.s | head -n 25

=== CÓDIGO ENSAMBLADOR x86_64 GENERADO (SÍNTESIS) ===
	.file	"hola_compilador.c"
	.intel_syntax noprefix
	.text
	.globl	sumar
	.type	sumar, @function
sumar:
.LFB0:
	.cfi_startproc
	endbr64
	push	rbp
	.cfi_def_cfa_offset 16
	.cfi_offset 6, -16
	mov	rbp, rsp
	.cfi_def_cfa_register 6
	mov	DWORD PTR -4[rbp], edi
	mov	DWORD PTR -8[rbp], esi
	mov	edx, DWORD PTR -4[rbp]
	mov	eax, DWORD PTR -8[rbp]
	add	eax, edx
	pop	rbp
	.cfi_def_cfa 7, 8
	ret
	.cfi_endproc
.LFE0:
	.size	sumar, .-sumar


## Compilación a código objeto
```bash
# Generación del código objeto (antes de enlazar) y conteo líneas
!gcc -c hola_compilador.c -o hola_compilador.o
!objdump -d hola_compilador.o | wc -l

In [ ]:
# Generación del código objeto (antes de enlazar) y conteo líneas
!gcc -c hola_compilador.c -o hola_compilador.o
!objdump -d hola_compilador.o | wc -l

60


## Compilación a código máquina ejecutable

```bash
# Generación del código ejecutable final (después de enlazar) y conteo líneas
!gcc hola_compilador.c -o hola_compilador
!objdump -d hola_compilador | wc -l


In [ ]:
# Generación del código ejecutable final (después de enlazar) y conteo líneas
!gcc hola_compilador.c -o hola_compilador
!objdump -d hola_compilador | wc -l

194


## El "Monstruo Completo" (Enlace Estático)  
Por defecto, GCC usa ***enlace dinámico*** usando una estructura de datos tabular, la tabla **PLT** (*Procedure Linkage Table*) ya que el código usa `printf`, el programa necesita conectarse con la biblioteca estándar de C (`libc.so`). El enlazador no copia el código de `printf` dentro del archivo para no duplicar espacio en el disco; en su lugar, crea un "trampolín" o acceso directo en una sección llamada `<printf@plt>`.

Cada vez que se invoca a `printf`, realmente salta a este bloque plt, el cual averigua en qué parte de la memoria RAM del sistema operativo se encuentra la verdadera función `printf` y redirige el control allí en tiempo de ejecución.

Para ver cómo el enlazador inyecta literalmente miles y miles de líneas copiando físicamente el código de `printf` y todas sus dependencias dentro del ejecutable, se debe compilar de forma estática:

```bash
# Compilación estática
!gcc -static hola_compilador.c -o hola_estatico

# Conteo de líneas de código ensamblador
!objdump -d hola_estatico | wc -l


In [ ]:
# Compilación estática
!gcc -static hola_compilador.c -o hola_estatico

# Conteo de líneas de código ensamblador
!objdump -d hola_estatico | wc -l

182934


## Ejecución del código
```bash
!echo ""
!echo "=== EJECUCIÓN DEL PROGRAMA OBJETO ==="
!./hola_compilador


In [ ]:
!echo ""
!echo "=== EJECUCIÓN DEL PROGRAMA OBJETO ==="
!./hola_compilador


=== EJECUCIÓN DEL PROGRAMA OBJETO ===
Escriba un numero entero: 5
La suma de 5 + 5 es: 10


## 4. Sección Práctica 2: Revelando el Front-End (Análisis Léxico y AST)
Para comprender cómo el compilador "desarma" el código fuente en tokens y estructuras jerárquicas, inspeccionaremos el compilador de Python a nivel interno.

### [CÓDIGO 2.1] Inspección del Analizador Léxico (Tokenizador)

```python
import token
import tokenize
from io import BytesIO

# Código fuente de prueba en formato cadena
codigo_fuente = "suma = a + 10"

# Tokenización del código fuente
tokens = tokenize.tokenize(BytesIO(codigo_fuente.encode("utf-8")).readline)

print(
    f"{'LINEA/COL':<12} | {'TIPO DE TOKEN':<20} | {'VALOR (LEXEMA)':<15}"
)
print("-" * 55)
for tok in tokens:
    if tok.type in (
        tokenize.ENCODING,
        tokenize.ENDMARKER,
        tokenize.NL,
        tokenize.NEWLINE,
    ):
        continue
    nombre_token = token.tok_name[tok.type]
    posicion = f"{tok.start[0]}:{tok.start[1]}"
    print(f"{posicion:<12} | {nombre_token:<20} | {tok.string:<15}")


In [ ]:
import token
import tokenize
from io import BytesIO

# Código fuente de prueba en formato cadena
codigo_fuente = "suma = a + 10"

# Tokenización del código fuente
tokens = tokenize.tokenize(BytesIO(codigo_fuente.encode("utf-8")).readline)

print(
    f"{'LINEA/COL':<12} | {'TIPO DE TOKEN':<20} | {'VALOR (LEXEMA)':<15}"
)
print("-" * 55)
for tok in tokens:
    if tok.type in (
        tokenize.ENCODING,
        tokenize.ENDMARKER,
        tokenize.NL,
        tokenize.NEWLINE,
    ):
        continue
    nombre_token = token.tok_name[tok.type]
    posicion = f"{tok.start[0]}:{tok.start[1]}"
    print(f"{posicion:<12} | {nombre_token:<20} | {tok.string:<15}")

LINEA/COL    | TIPO DE TOKEN        | VALOR (LEXEMA) 
-------------------------------------------------------
1:0          | NAME                 | suma           
1:5          | OP                   | =              
1:7          | NAME                 | a              
1:9          | OP                   | +              
1:11         | NUMBER               | 10             


### [CÓDIGO 2.2] Construcción del Árbol de Sintaxis Abstracta (AST)

```python
import ast

# Generar y visualizar la estructura sintáctica
arbol = ast.parse("suma = a + 10")
print("=== ÁRBOLES DE SINTAXIS ABSTRACTA (REPRESENTACIÓN EN TEXTO) ===")
print(ast.dump(arbol, indent=4))


In [ ]:
import ast

# Generar y visualizar la estructura sintáctica
arbol = ast.parse("suma = a + 10")
print("=== ÁRBOLES DE SINTAXIS ABSTRACTA (REPRESENTACIÓN EN TEXTO) ===")
print(ast.dump(arbol, indent=4))

=== ÁRBOLES DE SINTAXIS ABSTRACTA (REPRESENTACIÓN EN TEXTO) ===
Module(
    body=[
        Assign(
            targets=[
                Name(id='suma', ctx=Store())],
            value=BinOp(
                left=Name(id='a', ctx=Load()),
                op=Add(),
                right=Constant(value=10)))])


## 5. Sección Práctica 3: Inspección de la Tabla de Símbolos y Bytecode
En esta sección simularemos la interacción con la Tabla de Símbolos y observaremos la generación de código intermedio.

### [CÓDIGO 3.1] Simulación de una Tabla de Símbolos básica en Python
```python
class TablaDeSimbolos:

    def __init__(self):
        self.simbolos = {}

    def insertar(self, nombre, tipo, valor=None, ambito="global"):
        if nombre in self.simbolos:
            print(f"[ERROR LÉXICO/SINTÁCTICO] Identificador '{nombre}' ya declarado.")
        else:
            self.simbolos[nombre] = {
                "tipo": tipo,
                "valor": valor,
                "ambito": ambito,
            }
            print(f"[TABLA DE SÍMBOLOS] Insertado: {nombre} ({tipo})")

    def buscar(self, nombre):
        return self.simbolos.get(nombre, None)

    def mostrar(self):
        print("\n=== CONTENIDO DE LA TABLA DE SÍMBOLOS ===")
        print(f"{'NOMBRE':<12} | {'TIPO':<10} | {'ÁMBITO':<10} | {'VALOR':<10}")
        print("-" * 50)
        for nombre, datos in self.simbolos.items():
            print(
                f"{nombre:<12} | {datos['tipo']:<10} | {datos['ambito']:<10} | {str(datos['valor']):<10}"
            )


# Prueba de la Tabla de Símbolos
ts = TablaDeSimbolos()
ts.insertar("a", "ENTERO", 5)
ts.insertar("b", "ENTERO", 10)
ts.insertar("suma", "ENTERO", 15)
ts.mostrar()
```


In [ ]:
class TablaDeSimbolos:

    def __init__(self):
        self.simbolos = {}

    def insertar(self, nombre, tipo, valor=None, ambito="global"):
        if nombre in self.simbolos:
            print(f"[ERROR LÉXICO/SINTÁCTICO] Identificador '{nombre}' ya declarado.")
        else:
            self.simbolos[nombre] = {
                "tipo": tipo,
                "valor": valor,
                "ambito": ambito,
            }
            print(f"[TABLA DE SÍMBOLOS] Insertado: {nombre} ({tipo})")

    def buscar(self, nombre):
        return self.simbolos.get(nombre, None)

    def mostrar(self):
        print("\n=== CONTENIDO DE LA TABLA DE SÍMBOLOS ===")
        print(f"{'NOMBRE':<12} | {'TIPO':<10} | {'ÁMBITO':<10} | {'VALOR':<10}")
        print("-" * 50)
        for nombre, datos in self.simbolos.items():
            print(
                f"{nombre:<12} | {datos['tipo']:<10} | {datos['ambito']:<10} | {str(datos['valor']):<10}"
            )


# Prueba de la Tabla de Símbolos
ts = TablaDeSimbolos()
ts.insertar("a", "ENTERO", 5)
ts.insertar("b", "ENTERO", 10)
ts.insertar("suma", "ENTERO", 15)
ts.mostrar()

[TABLA DE SÍMBOLOS] Insertado: a (ENTERO)
[TABLA DE SÍMBOLOS] Insertado: b (ENTERO)
[TABLA DE SÍMBOLOS] Insertado: suma (ENTERO)

=== CONTENIDO DE LA TABLA DE SÍMBOLOS ===
NOMBRE       | TIPO       | ÁMBITO     | VALOR     
--------------------------------------------------
a            | ENTERO     | global     | 5         
b            | ENTERO     | global     | 10        
suma         | ENTERO     | global     | 15        


### [CÓDIGO 3.2] Desensamblado a Código Intermedio / Bytecode (Máquina de Pila)
```python
import dis

def calcular():
    a = 5
    b = 10
    suma = a + b
    return suma


print("=== BYTECODE GENERADO PARA LA MÁQUINA VIRTUAL DE PYTHON ===")
dis.dis(calcular)
```

In [ ]:
import dis

def calcular():
    a = 5
    b = 10
    suma = a + b
    return suma


print("=== BYTECODE GENERADO PARA LA MÁQUINA VIRTUAL DE PYTHON ===")
dis.dis(calcular)

=== BYTECODE GENERADO PARA LA MÁQUINA VIRTUAL DE PYTHON ===
  3           RESUME                   0

  4           LOAD_CONST               1 (5)
              STORE_FAST               0 (a)

  5           LOAD_CONST               2 (10)
              STORE_FAST               1 (b)

  6           LOAD_FAST_LOAD_FAST      1 (a, b)
              BINARY_OP                0 (+)
              STORE_FAST               2 (suma)

  7           LOAD_FAST                2 (suma)
              RETURN_VALUE


## 6. Desafío Asíncrono / Entregable de la Unidad I (Avance del Producto Integrador)

### Contexto del Proyecto Integrador Autónomo:
Durante el semestre, cada equipo concebirá, diseñará e implementará un **Analizador Léxico y Sintáctico (Compiler Front-End)** para un lenguaje original propuesto por el propio equipo.

Ejemplos de proyectos de semestres anteriores:
- **DSL para Configuración de Robots / Drones:** Lenguaje de comandos simples (`FORWARD 10`, `ROTATE 90`).
- **Lenguaje de Consultas para Grafos/Tablas:** Alternativa simplificada a SQL (`SELECT age FROM users WHERE status == 1`).
- **Mini-Lenguaje Matemático / Scripting:** Soporte para vectores, matrices o evaluación de expresiones lógicas/aritméticas.
- **Lenguaje Formato/Marcado Personalizado:** Generador de reportes en HTML/Markdown a partir de una sintaxis limpia.

---

### Instrucciones del Desafío U1 (Fase 0: Definición e Inspección Inicial):

#### Parte A: Documento de Especificación del Lenguaje (RFC del Equipo)
Crea un archivo `ESPECIFICACION.docx` que contenga:
1. **Nombre del Lenguaje Original y Propósito:** ¿Qué problema resuelve o a quién va dirigido?
2. **Ejemplo de Código Fuente Válido:** Muestra un fragmento de código de al menos 10-15 líneas escrito en tu nuevo lenguaje.
3. **Catálogo Preliminar de Tokens:** Lista las palabras reservadas, identificadores, constantes (enteras, flotantes, cadenas) y operadores que usará tu lenguaje.

#### Parte B: Prototipo de Inspección en Código (Colab Execution)
Utilizando las celdas mágicas `%%writefile` (en Python o C):
1. Crea un archivo con una cadena que contenga tu ejemplo de código fuente original.
2. Implementa una función de prueba preliminar (o utiliza la librería `tokenize` de Python como simulador) que tome la cadena de tu lenguaje y la desglose en una lista de componentes léxicos.
3. Muestra una estructura en código que sirva como **Tabla de Símbolos inicial** donde se registren las variables declaradas en tu lenguaje.

---

### Criterios Mínimos que Debe Cumplir Cualquier Lenguaje Propuesto (Checklist de Viabilidad):
Para que la propuesta sea aprobada por el catedrático, el lenguaje debe cumplir con:
- [ ] Poseer al menos **3 tipos de tokens bien diferenciados** (ej. Palabras Reservadas, Identificadores, Literales).
- [ ] Incluir soporte para **expresiones aritméticas o lógicas** anidadas.
- [ ] Incluir al menos una **estructura de control de flujo** (ej. `if/else`, `while`, `repeat`) o una **estructura de bloques** (funciones, comandos).
- [ ] Definir un mecanismo explícito de asignación o declaración de datos.

---

### Autoevaluación Muestreada (Comprobación Tipo Gradiance)
Responde las siguientes preguntas analizando la sintaxis de tu nuevo lenguaje y valida tus razonamientos en NotebookLM:

1. **Pregunta 1 (Conflictos Léxicos):** Al revisar las palabras reservadas y los identificadores de tu lenguaje, ¿existe alguna regla léxica que pudiera causar ambigüedad (por ejemplo, que una palabra reservada coincida con el patrón de un identificador de usuario)? ¿Cómo la resolverá tu analizador?
2. **Pregunta 2 (Estructura de la Tabla de Símbolos):** De los elementos de tu lenguaje original, ¿cuáles atributos (tipo, valor, ámbito, dirección de memoria) necesitará almacenar tu Tabla de Símbolos cuando se procese una declaración?

---

### Formato de Entrega / Portafolio de Evidencias
1. Guarda este cuaderno con todas las salidas ejecutadas (`Archivo` -> `Guardar una copia en GitHub` / `PEREZ JUAN PSSB I Colab U1.ipynb`).
2. Guarda el documento `ESPECIFICACION.docx` con la especificación de tu lenguaje en tu repositorio personal (no repositorios grupales) de GitHub.
3. Registra en **Moodle** el enlace del cuaderno ejecutable en Colab y el de tu repositorio conteniendo la especificación formal del proyecto.

## Sección de trabajo no dirigido
Prototipo de Inspección en Código (Colab Execution)
Utilizando las celdas mágicas `%%writefile` (en Python o C):
1. Crea un archivo con una cadena que contenga tu ejemplo de código fuente original.

In [ ]:
import os

pipespec_config = """pipeline "deploy-service":
stage "lint":
image = "golang:1.24"
exec:
"golangci-lint run"
stage "deploy" depends_on ["lint"]
image = "alpine/k8s:1.30"
timeout = 180
when = [[ branch == "main" ]]
exec:
"kubectl apply -f deploy/" """

file_name = "pipespec.conf"

with open(file_name, "w") as file:
    file.write(pipespec_config)

if os.path.exists(file_name):
  print(f"Archivo '{file_name}' creado con éxito.")
else:
  print(f"Error al crear archivo '{file_name}'.")


Archivo 'pipespec.conf' creado con éxito.


2. Implementa una función de prueba preliminar (o utiliza la librería `tokenize` de Python como simulador) que tome la cadena de tu lenguaje y la desglose en una lista de componentes léxicos.

In [ ]:
import os
import token
import tokenize
from io import BytesIO

file_name = "pipespec.conf"

if os.path.exists(file_name):
  with open(file_name, "r") as file:
    codigo_fuente = file.read()

  print(f"Archivo '{file_name}' cargado con éxito.\n")
  print("--- Contenido del archivo ---")
  print(codigo_fuente)

else:
    print(f"Error: '{file_name}' no encontrado.")


# Tokenización del código fuente
tokens = tokenize.tokenize(BytesIO(codigo_fuente.encode("utf-8")).readline)

print(
    f"{'\nLINEA/COL':<12} | {'TIPO DE TOKEN':<20} | {'VALOR (LEXEMA)':<15}"
)
print("-" * 55)
for tok in tokens:
    if tok.type in (
        tokenize.ENCODING,
        tokenize.ENDMARKER,
        tokenize.NL,
        tokenize.NEWLINE,
    ):
        continue
    nombre_token = token.tok_name[tok.type]
    posicion = f"{tok.start[0]}:{tok.start[1]}"
    print(f"{posicion:<12} | {nombre_token:<20} | {tok.string:<15}")

Archivo 'pipespec.conf' cargado con éxito.

--- Contenido del archivo ---
pipeline "deploy-service":
stage "lint":
image = "golang:1.24"
exec:
"golangci-lint run"
stage "deploy" depends_on ["lint"]
image = "alpine/k8s:1.30"
timeout = 180
when = [[ branch == "main" ]]
exec:
"kubectl apply -f deploy/" 

LINEA/COL   | TIPO DE TOKEN        | VALOR (LEXEMA) 
-------------------------------------------------------
1:0          | NAME                 | pipeline       
1:9          | STRING               | "deploy-service"
1:25         | OP                   | :              
2:0          | NAME                 | stage          
2:6          | STRING               | "lint"         
2:12         | OP                   | :              
3:0          | NAME                 | image          
3:6          | OP                   | =              
3:8          | STRING               | "golang:1.24"  
4:0          | NAME                 | exec           
4:4          | OP                   | :        

3. Muestra una estructura en código que sirva como **Tabla de Símbolos inicial** donde se registren las variables declaradas en tu lenguaje.

In [ ]:
import os
import token
import tokenize
from io import BytesIO

file_name = "pipespec.conf"

if os.path.exists(file_name):
    with open(file_name, "r") as file:
        codigo_fuente = file.read()
    print(f"Archivo '{file_name}' cargado con éxito.\n")
else:
    print(f"Error: '{file_name}' no encontrado.")
    codigo_fuente = ""

tokens = tokenize.tokenize(BytesIO(codigo_fuente.encode("utf-8")).readline)

class TablaDeSimbolos:
    def __init__(self):
        self.simbolos = {}

    def insertar(self, nombre, tipo, valor=None, ambito="global"):
        self.simbolos[nombre] = {
            "tipo": tipo,
            "valor": valor,
            "ambito": ambito,
        }

    def mostrar(self):
        print("\n=== CONTENIDO DE LA TABLA DE SÍMBOLOS ===")
        print(f"{'SÍMBOLO / TEXTO':<25} | {'TIPO TOKEN':<15} | {'VALOR':<15}")
        print("-" * 65)
        for nombre, datos in self.simbolos.items():
            print(
                f"{nombre:<25} | {datos['tipo']:<15} | {str(datos['valor']):<15}"
            )

ts = TablaDeSimbolos()

palabras_clave = {"pipeline", "stage", "image", "exec", "timeout", "when", "depends_on"}

for tok in tokens:
    if tok.type in (
        tokenize.ENCODING,
        tokenize.ENDMARKER,
        tokenize.NL,
        tokenize.NEWLINE,
        tokenize.INDENT,
        tokenize.DEDENT,
    ):
        continue

    nombre_token = token.tok_name[tok.type]
    texto_token = tok.string

    if texto_token in palabras_clave:
        tipo_simbolo = "PALABRA_CLAVE"
    elif tok.type == tokenize.NAME:
        tipo_simbolo = "IDENTIFICADOR"
    elif tok.type == tokenize.STRING:
        tipo_simbolo = "CADENA_TEXTO"
    elif tok.type == tokenize.NUMBER:
        tipo_simbolo = "NÚMERO"
    elif tok.type == tokenize.OP:
        tipo_simbolo = "OPERADOR"
    else:
        tipo_simbolo = nombre_token

    ts.insertar(nombre=texto_token, tipo=tipo_simbolo, valor=texto_token)

ts.mostrar()


Archivo 'pipespec.conf' cargado con éxito.


=== CONTENIDO DE LA TABLA DE SÍMBOLOS ===
SÍMBOLO / TEXTO           | TIPO TOKEN      | VALOR          
-----------------------------------------------------------------
pipeline                  | PALABRA_CLAVE   | pipeline       
"deploy-service"          | CADENA_TEXTO    | "deploy-service"
:                         | OPERADOR        | :              
stage                     | PALABRA_CLAVE   | stage          
"lint"                    | CADENA_TEXTO    | "lint"         
image                     | PALABRA_CLAVE   | image          
=                         | OPERADOR        | =              
"golang:1.24"             | CADENA_TEXTO    | "golang:1.24"  
exec                      | PALABRA_CLAVE   | exec           
"golangci-lint run"       | CADENA_TEXTO    | "golangci-lint run"
"deploy"                  | CADENA_TEXTO    | "deploy"       
depends_on                | PALABRA_CLAVE   | depends_on     
[                         | OPERADOR

1. **Pregunta 1 (Conflictos Léxicos):** Al revisar las palabras reservadas y los identificadores de tu lenguaje, ¿existe alguna regla léxica que pudiera causar ambigüedad (por ejemplo, que una palabra reservada coincida con el patrón de un identificador de usuario)? ¿Cómo la resolverá tu analizador?

**RESPUESTA:**

Si, una identificador ingresado por el usuario pudiera coincidir con una palabra reservada, en este caso, el analizador identificaría el valor del identificador y al compararlo con el valor fijo de la palabra reservada, este arrojaría un error de nombrado inválido para el identificador.

2. **Pregunta 2 (Estructura de la Tabla de Símbolos):** De los elementos de tu lenguaje original, ¿cuáles atributos (tipo, valor, ámbito, dirección de memoria) necesitará almacenar tu Tabla de Símbolos cuando se procese una declaración?

**RESPUESTA:**

Los atributos que se necesitarán almacenar en la Tabla de Símbolos al procesar una declaración serían: Identificador, tipo, asignación, valor y delimitador.